In [4]:
import sqlite3

conn = sqlite3.connect("enrollment.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    student_id INTEGER PRIMARY KEY AUTOINCREMENT,
    full_name TEXT NOT NULL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS courses (
    course_id INTEGER PRIMARY KEY,
    course_name TEXT NOT NULL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS enrollments (
    enrollment_id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id INTEGER,
    course_id INTEGER,
    UNIQUE(student_id, course_id),
    FOREIGN KEY(student_id) REFERENCES students(student_id),
    FOREIGN KEY(course_id) REFERENCES courses(course_id)
)
""")

students = [
    ("Anna Reyes",),
    ("Carlos Dela Cruz",),
    ("Mika Santos",),
    ("John Garcia",),
    ("Sarah Lopez",),
    ("Peter Cruz",),
    ("Angela Torres",),
    ("Kevin Tan",),
    ("Daniel Flores",),
    ("Maria Mendoza",)
]

courses = [
    (1, "Python Programming"),
    (2, "Database Systems"),
    (3, "Electronics"),
    (4, "Mathematics"),
    (5, "Computer Networks")
]

enrollments = [
    (1, 1), (1, 2),
    (2, 1), (2, 3),
    (3, 2), (3, 4),
    (4, 1), (4, 5),
    (5, 3), (5, 4),
    (6, 2), (6, 5),
    (7, 1), (7, 3),
    (8, 4), (8, 5),
    (9, 2), (9, 3),
    (10, 1), (10, 5)
]

cursor.executemany(
    "INSERT OR IGNORE INTO students (full_name) VALUES (?)",
    students
)

cursor.executemany(
    "INSERT OR IGNORE INTO courses (course_id, course_name) VALUES (?, ?)",
    courses
)

cursor.executemany(
    "INSERT OR IGNORE INTO enrollments (student_id, course_id) VALUES (?, ?)",
    enrollments
)

conn.commit()

while True:
    print("\nCOURSE ENROLLMENT TRACKER")
    print("1. Enroll Student")
    print("2. List Courses of a Student")
    print("3. List Students in a Course")
    print("4. Display All Students")
    print("5. Display All Courses")
    print("6. Exit")

    choice = input("Choose an option: ")

    if choice == "1":
        student_id = int(input("Enter Student ID: "))
        course_id = int(input("Enter Course ID: "))

        cursor.execute(
            "SELECT student_id FROM students WHERE student_id = ?",
            (student_id,)
        )
        student = cursor.fetchone()

        cursor.execute(
            "SELECT course_id FROM courses WHERE course_id = ?",
            (course_id,)
        )
        course = cursor.fetchone()

        if student and course:
            try:
                cursor.execute("""
                INSERT INTO enrollments (student_id, course_id)
                VALUES (?, ?)
                """, (student_id, course_id))

                conn.commit()
                print("Enrollment successful.")

            except sqlite3.IntegrityError:
                print("Warning: Student is already enrolled in this course.")
        else:
            print("Invalid student ID or course ID.")

    elif choice == "2":
        student_id = int(input("Enter Student ID: "))

        cursor.execute("""
        SELECT students.full_name, courses.course_name
        FROM enrollments
        JOIN students ON enrollments.student_id = students.student_id
        JOIN courses ON enrollments.course_id = courses.course_id
        WHERE students.student_id = ?
        """, (student_id,))

        results = cursor.fetchall()

        if results:
            for row in results:
                print(row)
        else:
            print("No enrollments found.")

    elif choice == "3":
        course_id = int(input("Enter Course ID: "))

        cursor.execute("""
        SELECT courses.course_name, students.full_name
        FROM enrollments
        JOIN courses ON enrollments.course_id = courses.course_id
        JOIN students ON enrollments.student_id = students.student_id
        WHERE courses.course_id = ?
        """, (course_id,))

        results = cursor.fetchall()

        if results:
            for row in results:
                print(row)
        else:
            print("No students enrolled in this course.")

    elif choice == "4":
        cursor.execute("SELECT * FROM students")
        for row in cursor.fetchall():
            print(row)

    elif choice == "5":
        cursor.execute("SELECT * FROM courses")
        for row in cursor.fetchall():
            print(row)

    elif choice == "6":
        print("Course Enrollment Tracker closed.")
        break

    else:
        print("Invalid choice. Please try again.")

conn.close()


COURSE ENROLLMENT TRACKER
1. Enroll Student
2. List Courses of a Student
3. List Students in a Course
4. Display All Students
5. Display All Courses
6. Exit
Enrollment successful.

COURSE ENROLLMENT TRACKER
1. Enroll Student
2. List Courses of a Student
3. List Students in a Course
4. Display All Students
5. Display All Courses
6. Exit
Course Enrollment Tracker closed.
